# Código de Huffman

### Autor: Prof. Lucas N. Alegre
### Adaptado de: https://www.geeksforgeeks.org/dsa/huffman-coding-in-python/

In [63]:
from typing import List, Dict
import heapq

In [64]:
def file_to_char_list(filename: str = "alice.txt"):
    """Dado um arquivo de texto, retorna uma lista de caracteres, convertendo símbolos UNICODE para símbolos correspondentes ASCII (0-127)"""
    with open(filename, mode="r", encoding="utf-8") as f:
        data = f.read()

    out  = []
    # processa a string de entrada, gerando uma lista de caracteres e convertendo símbolos UNICODE para símbolos correspondentes ASCII (0-127)
    for i in range(len(data)):
        s = data[i]
        if s == '”':
            out.append('"')
        elif  s == '“':
            out.append('"')
        elif s == '—':
            out.append('-')
        elif s == '’':
            out.append('\'')
        else:
            out.append(s)
    return out

def histogram(char_list: List[str]) -> Dict[str,int]:
    hist = {}
    for s in char_list:
        hist[s] = hist.get(s, 0) + 1
    return hist

In [65]:
char_list = file_to_char_list("alice.txt")

In [66]:
histogram(char_list)

{'A': 588,
 'l': 4679,
 'i': 6855,
 'c': 2278,
 'e': 13552,
 "'": 706,
 's': 6360,
 ' ': 24617,
 'd': 4779,
 'v': 837,
 'n': 6954,
 't': 10343,
 'u': 3460,
 'r': 5375,
 'W': 210,
 'o': 8072,
 'a': 8238,
 '\n': 3375,
 'b': 1401,
 'y': 2193,
 'L': 54,
 'w': 2468,
 'C': 143,
 'T': 385,
 'H': 221,
 'E': 69,
 'M': 176,
 'I': 688,
 'N': 74,
 'U': 16,
 'F': 65,
 'R': 93,
 'D': 161,
 'O': 88,
 '3': 1,
 '.': 999,
 '0': 1,
 'P': 74,
 'h': 7176,
 '-': 409,
 'f': 1938,
 'g': 2462,
 'V': 16,
 'S': 153,
 'B': 82,
 'm': 1934,
 'p': 1469,
 'Q': 83,
 'q': 129,
 'G': 72,
 'X': 8,
 'k': 1083,
 '?': 204,
 ',': 2427,
 ':': 233,
 '"': 2231,
 '(': 56,
 ')': 56,
 '_': 440,
 ';': 193,
 '!': 451,
 'j': 138,
 'x': 144,
 'Z': 1,
 'z': 77,
 'K': 76,
 '‘': 47,
 '*': 60,
 'J': 8,
 'Y': 72,
 'ù': 1,
 '[': 2,
 ']': 2}

In [67]:
class Node:
    """Nodo da árvore de Huffman. Pode ser um nó folha (com símbolo) ou um nó interno (sem símbolo, apenas frequência)"""
    def __init__(self, symbol=None, frequency=None):
        self.symbol = symbol
        self.frequency = frequency
        self.left = None
        self.right = None
        
    def __repr__(self, prefix="", is_left=True, code="") -> str:
        """Gera uma representação visual da árvore de Huffman, mostrando os códigos binários correspondentes a cada símbolo"""
        result = ""
        if self.right:
            result += self.right.__repr__(prefix + ("│   " if is_left else "    "), False, code=code+"1")

        label = f"'{self.symbol}  ({code})'"  if self.symbol is not None else f"({self.frequency})"
        result += prefix + ("└── " if is_left else "┌── ") + label + "\n"

        if self.left:
            result += self.left.__repr__(prefix + ("    " if is_left else "│   "), True, code=code+"0")
        return result

    def __lt__(self, other: Node) -> bool:
        """Define a comparação entre nós com base na frequência, para uso em uma fila de prioridade (heap)."""
        return self.frequency < other.frequency

def build_huffman_tree(hist: Dict[str,int]) -> Node:
    """Constrói a árvore de Huffman a partir do histograma de frequências dos caracteres.
    O(n log n), onde n é o número de caracteres distintos.
    """
    priority_queue = [Node(char, f) for char, f in hist.items()]
    heapq.heapify(priority_queue)

    while len(priority_queue) >= 2:
        left_child = heapq.heappop(priority_queue)
        right_child = heapq.heappop(priority_queue)

        merged_node = Node(frequency=left_child.frequency + right_child.frequency)
        merged_node.left = left_child
        merged_node.right = right_child

        heapq.heappush(priority_queue, merged_node)

    return priority_queue[0]

def generate_huffman_codes(node, code="", huffman_codes=None):
    """Gera os códigos de Huffman para cada símbolo na árvore."""
    if huffman_codes is None:
        huffman_codes = {}

    if node is not None:
        if node.symbol is not None:
            huffman_codes[node.symbol] = code

        generate_huffman_codes(node.left, code + "0", huffman_codes)
        generate_huffman_codes(node.right, code + "1", huffman_codes)

    return huffman_codes

In [68]:
root = build_huffman_tree(histogram(char_list))

In [69]:
root

│           ┌── '   (111)'
│       ┌── (48459)
│       │   │           ┌── '
  (110111)'
│       │   │       ┌── (6623)
│       │   │       │   │           ┌── '_  (110110111)'
│       │   │       │   │       ┌── (863)
│       │   │       │   │       │   │           ┌── '(  (110110110111)'
│       │   │       │   │       │   │       ┌── (112)
│       │   │       │   │       │   │       │   └── ')  (110110110110)'
│       │   │       │   │       │   │   ┌── (213)
│       │   │       │   │       │   │   │   │   ┌── 'L  (110110110101)'
│       │   │       │   │       │   │   │   └── (101)
│       │   │       │   │       │   │   │       └── '‘  (110110110100)'
│       │   │       │   │       │   └── (423)
│       │   │       │   │       │       └── 'W  (1101101100)'
│       │   │       │   │   ┌── (1700)
│       │   │       │   │   │   └── 'v  (11011010)'
│       │   │       │   └── (3248)
│       │   │       │       │       ┌── '-  (110110011)'
│       │   │       │       │   ┌── (806)
│ 

In [70]:
# Generate Huffman codes
huffman_codes = generate_huffman_codes(root)

In [71]:
huffman_codes

{'e': '000',
 'i': '0010',
 'n': '0011',
 'h': '0100',
 'u': '01010',
 '!': '01011000',
 'H': '010110010',
 ':': '010110011',
 '.': '0101101',
 'm': '010111',
 'o': '0110',
 'a': '0111',
 'f': '100000',
 'k': '1000010',
 ']': '100001100000000',
 '[': '100001100000001',
 '0': '1000011000000100',
 'ù': '1000011000000101',
 '3': '1000011000000110',
 'Z': '1000011000000111',
 'U': '1000011000001',
 'V': '1000011000010',
 'X': '10000110000110',
 'J': '10000110000111',
 '*': '10000110001',
 'q': '1000011001',
 'F': '10000110100',
 'E': '10000110101',
 'j': '1000011011',
 'C': '1000011100',
 'x': '1000011101',
 'G': '10000111100',
 'Y': '10000111101',
 'N': '10000111110',
 'P': '10000111111',
 'y': '100010',
 '"': '100011',
 'l': '10010',
 'c': '100110',
 ',': '100111',
 'd': '10100',
 'g': '101010',
 'w': '101011',
 't': '1011',
 'r': '11000',
 'A': '11001000',
 'K': '11001001000',
 'z': '11001001001',
 'S': '1100100101',
 'D': '1100100110',
 'B': '11001001110',
 'Q': '11001001111',
 'I': '1

In [72]:
size_original = 0
size_encoded = 0
for c in char_list:
    size_original += 8
    size_encoded += len(huffman_codes[c])
bytes_original = size_original / 8
bytes_encoded = size_encoded / 8
print("Tamanho do texto codificado:", bytes_encoded, "bytes")
print("Tamaho do texto em ASCII:", bytes_original, "bytes")
print("Compression ratio:", bytes_encoded/bytes_original)

Tamanho do texto codificado: 82855.125 bytes
Tamaho do texto em ASCII: 144581.0 bytes
Compression ratio: 0.5730706316874278


In [73]:
# Print Huffman codes
for char, code in huffman_codes.items():
    print(f"Character: {char}, Code: {code}")

Character: e, Code: 000
Character: i, Code: 0010
Character: n, Code: 0011
Character: h, Code: 0100
Character: u, Code: 01010
Character: !, Code: 01011000
Character: H, Code: 010110010
Character: :, Code: 010110011
Character: ., Code: 0101101
Character: m, Code: 010111
Character: o, Code: 0110
Character: a, Code: 0111
Character: f, Code: 100000
Character: k, Code: 1000010
Character: ], Code: 100001100000000
Character: [, Code: 100001100000001
Character: 0, Code: 1000011000000100
Character: ù, Code: 1000011000000101
Character: 3, Code: 1000011000000110
Character: Z, Code: 1000011000000111
Character: U, Code: 1000011000001
Character: V, Code: 1000011000010
Character: X, Code: 10000110000110
Character: J, Code: 10000110000111
Character: *, Code: 10000110001
Character: q, Code: 1000011001
Character: F, Code: 10000110100
Character: E, Code: 10000110101
Character: j, Code: 1000011011
Character: C, Code: 1000011100
Character: x, Code: 1000011101
Character: G, Code: 10000111100
Character: Y, Co

In [74]:
# Distribuições para teste
h1 = {'A': 60, 'B': 25, 'C': 10, 'D': 5}
h2 = {'A': 3, 'B': 2, 'C':  6, 'D': 8, 'E': 2, 'F': 6}
h3 = {'A': 45, 'B': 13, 'C': 12, 'D': 16, 'E': 9, 'F': 5}

In [75]:
root = build_huffman_tree(h3)
huffman_codes = generate_huffman_codes(root)
for char, code in huffman_codes.items():
    print(f"Character: {char}, Code: {code}")

Character: A, Code: 0
Character: C, Code: 100
Character: B, Code: 101
Character: F, Code: 1100
Character: E, Code: 1101
Character: D, Code: 111


In [76]:
root

│           ┌── 'D  (111)'
│       ┌── (30)
│       │   │   ┌── 'E  (1101)'
│       │   └── (14)
│       │       └── 'F  (1100)'
│   ┌── (55)
│   │   │   ┌── 'B  (101)'
│   │   └── (25)
│   │       └── 'C  (100)'
└── (100)
    └── 'A  (0)'